# GuardSLM — Open-Source SLM Safety Guard Evaluation (Google Colab)

This notebook contains pre-configured, ready-to-run execution blocks for **free open-source SLMs** on the **40 matched pairs / 80 test cases** dataset:

1. **Qwen 2.5 1.5B Instruct** (`Qwen/Qwen2.5-1.5B-Instruct`) — *Public, Ungated, Free HF*
2. **Qwen 2.5 7B Instruct** (`Qwen/Qwen2.5-7B-Instruct`) — *Public, Ungated, Free HF, 4-bit*
3. **Qwen 2.5 3B Instruct** (`Qwen/Qwen2.5-3B-Instruct`) — *Public, Ungated, Free HF*
4. **Qwen 2.5 0.5B Instruct** (`Qwen/Qwen2.5-0.5B-Instruct`) — *Public, Ungated, Ultra-fast SLM*
5. **Llama Guard 3 1B** (`meta-llama/Llama-Guard-3-1B`) — *Meta Gated (Requires free HF login token)*
6. **Rule Baseline** (Deterministic State Oracle)

### Step 1 — Clone / Update Repository

In [ ]:
import os
import sys

if os.path.exists('/content'):
    %cd /content
    if not os.path.exists('GuardSLM'):
        !git clone https://github.com/AkarshiAaryan/GuardSLM.git
    %cd GuardSLM
    !git pull

sys.path.insert(0, os.getcwd())
print(f"Project root directory: {os.getcwd()}")

### Step 2 — Install Free Open-Source Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q transformers torch accelerate bitsandbytes

### Step 3 — Hardware & VRAM Check

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM Capacity:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Running on CPU. Enable GPU acceleration: Runtime -> Change runtime type -> T4 GPU.")

### Step 4 — Load & Validate Test Cases

In [ ]:
from src.data.validator import validate_dataset_file
from src.data.loader import load_test_cases

dataset_path = 'data/dummy/dummy_cases.json'
is_valid, errors = validate_dataset_file(dataset_path)

if is_valid:
    cases = load_test_cases(dataset_path)
    print(f"SUCCESS: Loaded {len(cases)} test cases across {len(set(c.pair_id for c in cases))} matched pairs!")
else:
    print("Validation errors:", errors)

### Step 5A — Model 1: Qwen 2.5 1.5B (`Qwen/Qwen2.5-1.5B-Instruct`)

In [ ]:
from src.models.qwen_guard import QwenGuardAdapter
from src.evaluation.runner import run_evaluation_for_model
from src.evaluation.metrics import calculate_overall_metrics
from src.evaluation.pair_metrics import calculate_pair_metrics

qwen15_model = QwenGuardAdapter(name="qwen_1_5b", config={
    "enabled": True,
    "checkpoint": "Qwen/Qwen2.5-1.5B-Instruct",
    "load_in_4bit": False
})

qwen15_results = run_evaluation_for_model(qwen15_model, cases)
qwen15_overall = calculate_overall_metrics(qwen15_results)
qwen15_pair = calculate_pair_metrics(qwen15_results)

print(f"[Qwen 1.5B] Accuracy: {qwen15_overall['overall_accuracy']*100:.1f}% | Pair Accuracy: {qwen15_pair['pair_accuracy']*100:.1f}% | Flip Rate: {qwen15_pair['context_flip_rate']*100:.1f}%")

### Step 5B — Model 2: Qwen 2.5 7B (`Qwen/Qwen2.5-7B-Instruct` - 4-bit Quantized)

In [ ]:
qwen7b_model = QwenGuardAdapter(name="qwen_7b", config={
    "enabled": True,
    "checkpoint": "Qwen/Qwen2.5-7B-Instruct",
    "load_in_4bit": True  # Uses 4-bit quantization to fit GPU VRAM
})

try:
    qwen7b_results = run_evaluation_for_model(qwen7b_model, cases)
    qwen7b_overall = calculate_overall_metrics(qwen7b_results)
    qwen7b_pair = calculate_pair_metrics(qwen7b_results)
    print(f"[Qwen 7B] Accuracy: {qwen7b_overall['overall_accuracy']*100:.1f}% | Pair Accuracy: {qwen7b_pair['pair_accuracy']*100:.1f}%")
except Exception as e:
    print(f"Qwen 7B note: {e}")

### Step 5C — Model 3: Qwen 2.5 3B (`Qwen/Qwen2.5-3B-Instruct`)

In [ ]:
qwen3b_model = QwenGuardAdapter(name="qwen_3b", config={
    "enabled": True,
    "checkpoint": "Qwen/Qwen2.5-3B-Instruct",
    "load_in_4bit": False
})

try:
    qwen3b_results = run_evaluation_for_model(qwen3b_model, cases)
    qwen3b_overall = calculate_overall_metrics(qwen3b_results)
    qwen3b_pair = calculate_pair_metrics(qwen3b_results)
    print(f"[Qwen 3B] Accuracy: {qwen3b_overall['overall_accuracy']*100:.1f}% | Pair Accuracy: {qwen3b_pair['pair_accuracy']*100:.1f}%")
except Exception as e:
    print(f"Qwen 3B note: {e}")

### Step 5D — Model 4: Qwen 2.5 0.5B (`Qwen/Qwen2.5-0.5B-Instruct` - Ultra-fast SLM Sweep)

In [ ]:
qwen05b_model = QwenGuardAdapter(name="qwen_0_5b", config={
    "enabled": True,
    "checkpoint": "Qwen/Qwen2.5-0.5B-Instruct",
    "load_in_4bit": False
})

try:
    qwen05b_results = run_evaluation_for_model(qwen05b_model, cases)
    qwen05b_overall = calculate_overall_metrics(qwen05b_results)
    qwen05b_pair = calculate_pair_metrics(qwen05b_results)
    print(f"[Qwen 0.5B] Accuracy: {qwen05b_overall['overall_accuracy']*100:.1f}% | Pair Accuracy: {qwen05b_pair['pair_accuracy']*100:.1f}%")
except Exception as e:
    print(f"Qwen 0.5B note: {e}")

### Step 5E — Model 5: Llama Guard 3 1B (`meta-llama/Llama-Guard-3-1B` - Optional Gated Login)

In [ ]:
from src.models.llama_guard import LlamaGuardAdapter

# Optional: If you want to evaluate Meta's gated Llama Guard, log in with your free HF token:
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN")

llama_model = LlamaGuardAdapter(name="llama_guard", config={
    "enabled": True,
    "checkpoint": "meta-llama/Llama-Guard-3-1B",
    "load_in_4bit": False
})

try:
    llama_results = run_evaluation_for_model(llama_model, cases)
    llama_overall = calculate_overall_metrics(llama_results)
    llama_pair = calculate_pair_metrics(llama_results)
    print(f"[Llama Guard 3] Accuracy: {llama_overall['overall_accuracy']*100:.1f}% | Pair Accuracy: {llama_pair['pair_accuracy']*100:.1f}%")
except Exception as e:
    print(f"Llama Guard note: {e}")

### Step 5F — Model 6: Deterministic State Rule Baseline (`RuleBaseline`)

In [ ]:
from src.baselines.rule_baseline import RuleBaseline

rule_model = RuleBaseline(name="rule_baseline")
rule_results = run_evaluation_for_model(rule_model, cases)
rule_overall = calculate_overall_metrics(rule_results)
rule_pair = calculate_pair_metrics(rule_results)

print(f"[Rule Baseline] Accuracy: {rule_overall['overall_accuracy']*100:.1f}% | Pair Accuracy: {rule_pair['pair_accuracy']*100:.1f}%")

### Step 6 — Generate & Display Decision Gate Report

In [ ]:
from IPython.display import display, Markdown
from src.evaluation.failure_analysis import analyze_failures
from src.evaluation.decision_gate import generate_decision_gate_report

failures = analyze_failures(qwen15_results)
report_path = 'reports/colab_decision_gate_report.md'
report_content = generate_decision_gate_report(qwen15_overall, qwen15_pair, failures, report_path)

print(f"Decision Gate Report generated successfully at: {report_path}\n")
display(Markdown(report_content))